### Searchlight
scanning the brain with a searchlight. Briefly, a ball of given radius is scanned across the brain volume and the prediction accuracy of a classifier trained on the corresponding voxels is measured.

In [ ]:
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}

ELECTRODE_TO_REGION = {
    electrode: region 
    for region, electrodes in REGION_TO_ELECTRODES.items() 
    for electrode in electrodes
}

REGION_TO_LABEL = {
    'Middle_Finger': 0,
    'Hand': 1,
    'Forearm': 2,
    'Arm': 3
}

LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

print("Region-to-Electrode Mapping:")
for region, electrodes in REGION_TO_ELECTRODES.items():
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {electrodes} ({len(electrodes)} electrodes)")

In [ ]:
import json
from pathlib import Path

BIDS_ROOT = Path(r"../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"

subjects = ["sub-p0002"]
session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs_per_subject = 4


HRF_DELAY = 6.0
WINDOW = 1

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])
print(f"TR: {tr} s")

RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_LOG = RESULTS_DIR / "4_Classes_ANN_analysis_log.txt"
log_file = open(OUTPUT_LOG, 'w', encoding='utf-8')
experiments_results = []

In [ ]:
import numpy as np

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
import pandas as pd

all_events = []
subject = subjects[0]
for run in range(1, n_runs_per_subject + 1):
    events_path = BIDS_ROOT / subject / session / "func" / f"{subject}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(events_path, sep='\t')
    df['subject'] = subject
    df['run'] = run
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()

#eletrode -> region mapping
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)

In [ ]:
region_counts = stim_events['region'].value_counts()
for region in ['Middle_Finger', 'Hand', 'Forearm', 'Arm']:
    count = region_counts.get(region, 0)
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {count} samples")

In [ ]:
from nilearn.image import load_img, index_img

first_run_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii")
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)
print(f"\nReference image shape: {first_run_img.shape}")

In [ ]:
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.image import new_img_like, resample_to_img
from nilearn.maskers import NiftiMasker
from nilearn.plotting import plot_roi

atlas = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
mask_data = np.isin(atlas_data, s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')
masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)
display = plot_roi(s1_mask_resampled, bg_img=ref_img, 
                            title='S1 ROI Mask (Left G_postcentral)',
                            display_mode='ortho', cut_coords=(0, -20, 60))
display.savefig(FIGURES_DIR / 's1_mask.png', dpi=150)
display.close()

In [ ]:
from nilearn.image import concat_imgs

volumes = []
y_list = []
run_labels = []

for run in range(1, n_runs_per_subject + 1):
    bold_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii")
    bold_img = load_img(str(bold_path))
    print(f"Run {run}...")
    run_events = stim_events[stim_events['run'] == run]
    for _, event in run_events.iterrows():
        onset = event["onset"]
        region = event["region"]
        target_time = onset + HRF_DELAY
        vol_indx = int(np.round(target_time / tr))
        
        vol_indices = [vol_indx - WINDOW, vol_indx, vol_indx + WINDOW]
        if all(0 <= v < bold_img.shape[3] for v in vol_indices):
            imgs = [index_img(bold_img, v) for v in vol_indices]
            avg_data = np.mean([img.get_fdata() for img in imgs], axis=0)
            avg_img = new_img_like(imgs[0], avg_data)
            volumes.append(avg_img)
            y_list.append(REGION_TO_LABEL[region])
            run_labels.append(run)

In [ ]:
fmri_4d = concat_imgs(volumes)
y = np.array(y_list)
groups = np.array(run_labels)

print(f"\n4D fMRI shape: {fmri_4d.shape}")
print(f"Labels: {np.unique(y, return_counts=True)}")
print(f"Groups: {np.unique(groups, return_counts=True)}")

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import LeaveOneGroupOut
from nilearn.decoding import SearchLight

estimator = LinearSVC(C=3, max_iter=10000)
cv = LeaveOneGroupOut()

In [ ]:
search_light = SearchLight(
    mask_img=s1_mask_resampled,
    process_mask_img=s1_mask_resampled,
    radius=4, #Kriegeskorte et al. (2006), 4mm, each sphere contains roughly 30 voxels
    estimator=estimator,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
)
search_light.fit(fmri_4d, y, groups=groups)

In [ ]:
from nilearn.image import mean_img

mean_fmri = mean_img(fmri_4d)
scores_img = search_light.scores_img_
scores_data = scores_img.get_fdata()
mask_bool = s1_mask_resampled.get_fdata().astype(bool)
scores_in_s1 = scores_data[mask_bool]

In [ ]:
print("Searchlight scores in S1:")
print(f"  Mean:  {scores_in_s1.mean():.4f}")
print(f"  Std:   {scores_in_s1.std():.4f}")
print(f"  Max:   {scores_in_s1.max():.4f}")
print(f"  Voxels above chance (>0.25): {(scores_in_s1 > 0.25).sum()} / {len(scores_in_s1)}")
print(f"  Voxels above 0.40: {(scores_in_s1 > 0.40).sum()} / {len(scores_in_s1)}")
print(f"  Voxels above 0.50: {(scores_in_s1 > 0.50).sum()} / {len(scores_in_s1)}")

In [ ]:
from nilearn.plotting import plot_img, show

display = plot_img(
    scores_img,
    bg_img=mean_fmri,
    title="Searchlight Accuracy (4-class, S1, radius=4mm)",
    display_mode="ortho",
    cut_coords=(0, -20, 60),
    vmin=0.0,
    vmax=0.8,
    cmap="inferno",
    threshold=0.25,
    colorbar=True,
    black_bg=True,
)
display.savefig(FIGURES_DIR / 'searchlight_scores_ortho.png', dpi=150)
show()

log_file.close()

In [ ]:
from nilearn.plotting import view_img_on_surf
top_threshold = np.percentile(scores_in_s1, 75)

view_top = view_img_on_surf(
    scores_img,
    surf_mesh='fsaverage',
    title=f"Searchlight Top 25% (threshold={top_threshold:.3f})",
    symmetric_cmap=False,
    vmin=top_threshold,
    vmax=scores_in_s1.max(),
    threshold=top_threshold,
    vol_to_surf_kwargs={"interpolation":"nearest_most_frequent"}
)
view_top